# Imports

In [61]:
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import networkx as nx

# Load Dataset

In [62]:
DATASET_DIR = Path("dataset")

alerts = [
    json.loads(line)
    for line in open(
        DATASET_DIR / "alerts_sample.jsonl",
        encoding="utf-8"
    )
]

services_data = json.load(
    open(
        DATASET_DIR / "services.json",
        encoding="utf-8"
    )
)

print("alerts:", len(alerts))
print("services:", len(services_data["services"]))
print("edges:", len(services_data["edges"]))

alerts: 20
services: 10
edges: 17


# Fingerprint + Dedup

In [63]:
def fingerprint(alert):
    return (
        f"{alert['service']}|"
        f"{alert['metric']}|"
        f"{alert['severity']}"
    )


def dedup_alerts(alerts):
    groups = {}

    for alert in alerts:
        fp = fingerprint(alert)

        if fp not in groups:
            groups[fp] = {
                "fingerprint": fp,
                "count": 0,
                "alerts": []
            }

        groups[fp]["count"] += 1
        groups[fp]["alerts"].append(alert)

    return groups

In [64]:
dedup = dedup_alerts(alerts)

print("raw alerts:", len(alerts))
print("unique fingerprints:", len(dedup))

for fp, data in list(dedup.items())[:5]:
    print(fp, "count =", data["count"])

raw alerts: 20
unique fingerprints: 17
payment-svc|db_connection_pool_used_ratio|warn count = 1
payment-svc|db_connection_pool_used_ratio|crit count = 2
payment-svc|latency_p99_ms|crit count = 3
payment-svc|error_rate|warn count = 1
checkout-svc|latency_p99_ms|warn count = 1


# Build Service Graph

In [65]:
def build_graph(data):
    g = nx.DiGraph()

    for svc in data["services"]:
        g.add_node(svc["name"], kind="service")

    for store in data["stores"]:
        g.add_node(store["name"], kind="store")

    for edge in data["edges"]:
        g.add_edge(
            edge["from"],
            edge["to"],
            edge_type=edge["type"]
        )

    return g


graph = build_graph(services_data)

print(
    "nodes:",
    graph.number_of_nodes(),
    "edges:",
    graph.number_of_edges()
)

nodes: 14 edges: 17


# Session Window

In [66]:
def parse_ts(ts):
    return datetime.fromisoformat(
        ts.replace("Z", "+00:00")
    )


def session_groups(alerts, gap_sec=120):

    alerts = sorted(
        alerts,
        key=lambda a: parse_ts(a["ts"])
    )

    if not alerts:
        return []

    sessions = [[alerts[0]]]

    for alert in alerts[1:]:

        current_ts = parse_ts(alert["ts"])

        previous_ts = parse_ts(
            sessions[-1][-1]["ts"]
        )

        gap = (
            current_ts - previous_ts
        ).total_seconds()

        if gap <= gap_sec:
            sessions[-1].append(alert)
        else:
            sessions.append([alert])

    return sessions

# Path-Based Topology Grouping

In [67]:
def topology_group(
    alerts,
    graph,
    max_hop=2
):

    if not alerts:
        return []

    undirected = graph.to_undirected()

    by_service = defaultdict(list)

    for a in alerts:
        by_service[a["service"]].append(a)

    services = list(by_service.keys())

    parent = {
        s: s
        for s in services
    }

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        parent[find(a)] = find(b)

    for i, s1 in enumerate(services):

        for s2 in services[i + 1:]:

            try:

                dist = nx.shortest_path_length(
                    undirected,
                    s1,
                    s2
                )

                if dist <= max_hop:
                    union(s1, s2)

            except nx.NetworkXNoPath:
                pass

    groups = defaultdict(list)

    for svc in services:
        groups[find(svc)].extend(
            by_service[svc]
        )

    return list(groups.values())

# Correlation Pipeline

In [68]:
SEVERITY_RANK = {
    "info": 0,
    "warn": 1,
    "crit": 2
}

NOISE_KEYWORDS = [
    "noise",
    "unrelated"
]


def is_noise_alert(alert):
    note = (
        alert.get("labels", {})
             .get("note", "")
             .lower()
    )

    return any(
        keyword in note
        for keyword in NOISE_KEYWORDS
    )


def build_cluster(cluster_id, group):

    max_sev = max(
        group,
        key=lambda a: SEVERITY_RANK.get(
            a["severity"], 0
        )
    )["severity"]

    return {
        "cluster_id": cluster_id,

        "alert_count": len(group),

        "services": sorted(
            set(
                a["service"]
                for a in group
            )
        ),

        "time_range": [
            min(a["ts"] for a in group),
            max(a["ts"] for a in group)
        ],

        "max_severity": max_sev,

        "alert_ids": [
            a["id"]
            for a in group
        ],

        "fingerprints": sorted(
            set(
                fingerprint(a)
                for a in group
            )
        )
    }


def correlate(
    alerts,
    graph,
    gap_sec=120,
    max_hop=2
):

    sessions = session_groups(
        alerts,
        gap_sec
    )

    clusters = []
    cluster_no = 1

    for session in sessions:

        # -----------------------------
        # Separate obvious noise alerts
        # -----------------------------
        noise_alerts = []
        incident_alerts = []

        for alert in session:

            if is_noise_alert(alert):
                noise_alerts.append(alert)
            else:
                incident_alerts.append(alert)

        # -----------------------------
        # Normal topology grouping
        # -----------------------------
        topo_groups = topology_group(
            incident_alerts,
            graph,
            max_hop=max_hop
        )

        for group in topo_groups:

            clusters.append(
                build_cluster(
                    f"c-{cluster_no:03d}",
                    group
                )
            )

            cluster_no += 1

        # -----------------------------
        # Each noise alert = own cluster
        # -----------------------------
        for alert in noise_alerts:

            clusters.append(
                build_cluster(
                    f"c-{cluster_no:03d}",
                    [alert]
                )
            )

            cluster_no += 1

    return clusters

In [69]:
clusters = correlate(
    alerts,
    graph,
    gap_sec=120,
    max_hop=2
)

print("input alerts:", len(alerts))
print("clusters:", len(clusters))

for c in clusters:
    print(
        c["cluster_id"],
        c["alert_count"],
        c["services"]
    )

input alerts: 20
clusters: 3
c-001 18 ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
c-002 1 ['recommender-svc']
c-003 1 ['search-svc']


# Save Result

In [70]:
output = {
    "input_alerts": len(alerts),
    "output_clusters": len(clusters),
    "reduction_ratio":
        1 - (
            len(clusters)
            / len(alerts)
        ),
    "clusters": clusters
}

Path("results").mkdir(
    exist_ok=True
)

with open(
    "results/cluster_summary.json",
    "w"
) as f:
    json.dump(
        output,
        f,
        indent=2
    )

print(
    "saved:",
    "results/cluster_summary.json"
)

saved: results/cluster_summary.json
